In [1]:
# =========================
# LIMPIEZA DE ENTORNO
# =========================

!pip uninstall -y tensorflow tensorflow-cpu tensorflow-gpu protobuf transformers peft accelerate -q

# =========================
# INSTALACIÓN BASE (COMPATIBLE)
# =========================

!pip install transformers==4.38.2 accelerate==0.27.2 protobuf==3.20.3 -q

# =========================
# MÉTRICAS
# =========================

!pip install evaluate bert-score sacrebleu unbabel-comet bert-score -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 130.7/130.7 kB 7.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.5/8.5 MB 163.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 280.0/280.0 kB 38.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 162.1/162.1 kB 28.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 69.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 175.4 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
ydf-tf 2.20.0 requires tensorflow==2.20.0, which is not installed.
google-cloud-firestore 2.27.0 requires protobuf<8.0.0,>=4.25.8, but you have protobuf 3.20.3 which is incompatible.
google-cloud-bigtable 2.38.0 requires protobuf<8.0.0,>=4.25.8, but you have protobuf 3.20.3 which is incompatible.
tensorflow-metadata 1.21.0 requires protobuf>=4

In [1]:
import pandas as pd
from google.colab import drive
drive.mount('/content/drive')

path = "/content/drive/MyDrive/Tesis 2/Entrenamiento modelos NMT/Shipibo Konibo/train.csv"

df = pd.read_csv(path)
df = df[["source", "target"]].dropna()

Mounted at /content/drive


In [2]:
import re

def limpiar_texto(texto):
    return re.sub(r"[^\w\s-]", "", texto)

In [3]:
#Aplicamos limpieza

df["source_clean"] = df["source"].apply(limpiar_texto)
df["target_clean"] = df["target"].apply(limpiar_texto)

# SILABIFICADOR

In [4]:
VOWELS = {'a', 'e', 'i', 'o'}

DIGRAPHS = {'ch', 'sh', 'ts'}

CODAS = {'n', 's', 'sh', 'x'}

def tokenize(word):
    tokens = []
    i = 0

    while i < len(word):

        if i + 1 < len(word) and word[i:i+2] in DIGRAPHS:
            tokens.append(word[i:i+2])
            i += 2
        else:
            tokens.append(word[i])
            i += 1

    return tokens

def silabificador_shipibo(word):

    tokens = tokenize(word)

    syllables = []
    current = []

    i = 0

    while i < len(tokens):

        current.append(tokens[i])

        # si es vocal
        if tokens[i] in VOWELS:

            # mirar siguiente
            if i + 1 < len(tokens):

                nxt = tokens[i + 1]

                # caso VCV
                if nxt not in VOWELS:

                    # mirar siguiente del siguiente
                    if i + 2 < len(tokens):

                        nxt2 = tokens[i + 2]

                        # VCV → dividir antes de consonante
                        if nxt2 in VOWELS:
                            syllables.append(current)
                            current = []

                        # VCCV
                        else:

                            # coda válida
                            if nxt in CODAS:
                                current.append(nxt)
                                syllables.append(current)
                                current = []
                                i += 1
                            else:
                                syllables.append(current)
                                current = []

                    else:
                        # final
                        if nxt in CODAS:
                            current.append(nxt)
                            i += 1

            syllables.append(current)
            current = []

        i += 1

    if current:
        syllables.append(current)

    return [''.join(s) for s in syllables]

In [5]:
def silabificar_texto(texto):
    """
    El silabificador trabaja palabra por palabra.

    Cada palabra:
    palabra → sílabas

    Luego todas las sílabas se unen
    en una sola secuencia.
    """

    palabras = texto.split()

    resultado = []

    for palabra in palabras:

        silabas = silabificador_shipibo(palabra)

        resultado.extend(silabas)

    return resultado

In [6]:
# =========================
# EXTRAER SUBWORDS Y SÍLABAS
# =========================
from collections import Counter
subword_counter = Counter()
for texto in df["source_clean"]:
    palabras = texto.split()
    for palabra in palabras:
        silabas = silabificador_shipibo(palabra)
        subword_counter.update(silabas)

print("SUBWORDS MÁS FRECUENTES:")
print(subword_counter.most_common(50))


# ==============================================================================
# FILTRAR TOKENS ÚTILES DEL SILABIFICADOR (UMBRAL >= 50)
# ==============================================================================
nuevos_tokens = [token for token, count in subword_counter.items() if count >= 50]
print("Cantidad de nuevos tokens a agregar (freq >= 50):", len(nuevos_tokens))


SUBWORDS MÁS FRECUENTES:
[('', 283468), ('i', 49358), ('ja', 35675), ('a', 34753), ('ti', 19882), ('ka', 19741), ('ki', 19239), ('bo', 16370), ('ra', 16368), ('ma', 14632), ('na', 13157), ('bi', 12238), ('yo', 8828), ('ko', 8499), ('to', 8457), ('ke', 7977), ('we', 7876), ('in', 7871), ('o', 7640), ('xon', 7595), ('kin', 7232), ('ba', 7058), ('no', 6979), ('ri', 6827), ('ni', 6104), ('nan', 5949), ('an', 5824), ('jo', 5525), ('jas', 5192), ('ta', 5086), ('ya', 4656), ('tan', 4628), ('e', 4593), ('kon', 4572), ('shi', 4012), ('me', 3892), ('be', 3509), ('pa', 3377), ('non', 3343), ('kan', 3315), ('te', 3266), ('ton', 3131), ('ne', 3088), ('on', 3026), ('di', 2625), ('os', 2533), ('ká', 2506), ('pi', 2332), ('sha', 2286), ('mo', 2129)]
Cantidad de nuevos tokens a agregar (freq >= 50): 262


In [7]:
for i in range(5):

    print("ORIGINAL:", df.iloc[i]["source"])

    print("LIMPIO:", df.iloc[i]["source_clean"])

    print("TARGET:", df.iloc[i]["target_clean"])

    print("-----")

ORIGINAL: ¿matoki wishabo texea?
LIMPIO: matoki wishabo texea
TARGET: le sobran letras
-----
ORIGINAL: kenawe ainbo bakebo itan benbo bakebo chosko tsinki akanti
LIMPIO: kenawe ainbo bakebo itan benbo bakebo chosko tsinki akanti
TARGET: invita a las niñas y a los niños a formar grupos de cuatro
-----
ORIGINAL: moa wetsa baritian
LIMPIO: moa wetsa baritian
TARGET: en el año anterior
-----
ORIGINAL: shinanwe itan yoiwe jawekeska axonmein min jaska axon toponti shinanbo pikoti iki.
LIMPIO: shinanwe itan yoiwe jawekeska axonmein min jaska axon toponti shinanbo pikoti iki
TARGET: razona y argumenta generando ideas matemáticas
-----
ORIGINAL: shinan awe benbo bakebo itan ainbo bakebo betan jawekopiki min jaskara shinanbo akai ixon
LIMPIO: shinan awe benbo bakebo itan ainbo bakebo betan jawekopiki min jaskara shinanbo akai ixon
TARGET: establece con los niños y las niñas el propósito de los mensajes que van a escribir
-----


***Creación de dataset***

In [8]:
from datasets import Dataset
df_model = df[["source_clean", "target_clean"]].rename(
    columns={
        "source_clean": "source",
        "target_clean": "target"
    }
)
dataset = Dataset.from_pandas(df_model)
print(dataset)


Dataset({
    features: ['source', 'target'],
    num_rows: 14728
})


In [9]:
print(dataset[0])

{'source': 'matoki wishabo texea', 'target': 'le sobran letras'}


***Tokenización***

In [10]:
##Llamar al tokenizador
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(
    "facebook/nllb-200-distilled-600M"
)


# ==============================================================================
# AGREGAR TOKENS AL TOKENIZER (Aquí se agregan los nuevos tokens al tokenizador)
# ==============================================================================
tokens_filtrados = [token for token in nuevos_tokens if token not in tokenizer.get_vocab()]
print("Nuevos tokens reales a agregar:", len(tokens_filtrados))
tokens_agregados = tokenizer.add_tokens(tokens_filtrados)
print("Tokens agregados con éxito:", tokens_agregados)


/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/564 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/4.85M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.3M [00:00<?, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

Nuevos tokens reales a agregar: 65
Tokens agregados con éxito: 64


In [11]:
tokenizer.src_lang = "quy_Latn"
tokenizer.tgt_lang = "spa_Latn"

In [12]:
##Probar tokenización

example = dataset[0]["source"]
example2 = dataset[0]["target"]

print("Texto:")
print(example)

tokens = tokenizer(example)
tokens2 = tokenizer(example2)

print("\nTokens:")
print(tokens["input_ids"])
print(tokenizer.convert_ids_to_tokens(tokens["input_ids"]))
print(tokens2["input_ids"])

Texto:
matoki wishabo texea

Tokens:
[256144, 601, 1930, 40, 2632, 298, 20236, 1994, 2]
['quy_Latn', '▁mat', 'oki', '▁w', 'isha', 'bo', '▁tex', 'ea', '</s>']
[256144, 96, 3056, 823, 240182, 2]


In [13]:
## Función para tokenizar todo el dataset

def tokenizar_ejemplo(example):
    model_inputs = tokenizer(
        example["source"],
        max_length=64,
        truncation=True,
        padding="max_length"
    )

    labels = tokenizer(
        example["target"],
        max_length=64,
        truncation=True,
        padding="max_length"
    )

    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

In [14]:
##Tokenizar todo el dataset

tokenized_dataset = dataset.map(tokenizar_ejemplo)

Map:   0%|          | 0/14728 [00:00<?, ? examples/s]

In [15]:
# Parche de compatibilidad: torchvision en Colab puede no tener VideoReader
try:
    import torchvision.io
    if not hasattr(torchvision.io, 'VideoReader'):
        class _DummyVideoReader:
            pass
        torchvision.io.VideoReader = _DummyVideoReader
except ImportError:
    pass

##Limpiar el dataset de columnas iniciales

tokenized_dataset = tokenized_dataset.remove_columns(["source", "target"])
tokenized_dataset.set_format("torch")

In [16]:
import torch

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Usando:", device)

Usando: cuda


***Modelo***

In [17]:
##Llamar al modelo
from transformers import AutoModelForSeq2SeqLM

model = AutoModelForSeq2SeqLM.from_pretrained(
    "facebook/nllb-200-distilled-600M"
)

model.to(device)
# =========================
# REDIMENSIONAR EMBEDDINGS
# =========================
model.resize_token_embeddings(len(tokenizer))


/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


config.json:   0%|          | 0.00/846 [00:00<?, ?B/s]

/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


pytorch_model.bin:   0%|          | 0.00/2.46G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/189 [00:00<?, ?B/s]

Embedding(256268, 1024)

In [18]:
#Parámetros para el entrenamiento
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="./results_nllb",
    per_device_train_batch_size=1,
    num_train_epochs=3,
    learning_rate=5e-5,
    save_strategy="epoch",
    fp16=True
)

In [19]:
##Creación del trainer

from transformers import Trainer

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset
)

/usr/local/lib/python3.12/dist-packages/accelerate/accelerator.py:450: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  self.scaler = torch.cuda.amp.GradScaler(**kwargs)


In [20]:
#Entrenar el modelo
trainer.train()

/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: (1) Create a W&B account
wandb: (2) Use an existing W&B account
wandb: (3) Don't visualize my results


wandb: Enter your choice: 3


wandb: You chose "Don't visualize my results"
wandb: Using W&B in offline mode.
wandb: W&B API key is configured. Use `wandb login --relogin` to force relogin


Step,Training Loss
500,3.529200
1000,1.185600
1500,1.149600
2000,1.086200
2500,1.036300
3000,1.016100
3500,1.018200
4000,0.991200
4500,0.999800
5000,0.981300


Some non-default generation parameters are set in the model config. These should go into a GenerationConfig file (https://huggingface.co/docs/transformers/generation_strategies#save-a-custom-decoding-strategy-with-your-model) instead. This warning will be raised to an exception in v4.41.
Non-default generation parameters: {'max_length': 200}
Some non-default generation parameters are set in the model config. These should go into a GenerationConfig file (https://huggingface.co/docs/transformers/generation_strategies#save-a-custom-decoding-strategy-with-your-model) instead. This warning will be raised to an exception in v4.41.
Non-default generation parameters: {'max_length': 200}
Some non-default generation parameters are set in the model config. These should go into a GenerationConfig file (https://huggingface.co/docs/transformers/generation_strategies#save-a-custom-decoding-strategy-with-your-model) instead. This warning will be raised to an exception in v4.41.
Non-default generation 

TrainOutput(global_step=44184, training_loss=0.7138568996282326, metrics={'train_runtime': 3319.5602, 'train_samples_per_second': 13.31, 'train_steps_per_second': 13.31, 'total_flos': 5984459358732288.0, 'train_loss': 0.7138568996282326, 'epoch': 3.0})

In [21]:
#Guardar el modelo

trainer.save_model("/content/drive/MyDrive/Tesis 2/Entrenamiento modelos NMT/Shipibo Konibo/modelo_NLLB_ShipiboKonibo_v4")
tokenizer.save_pretrained("/content/drive/MyDrive/Tesis 2/Entrenamiento modelos NMT/Shipibo Konibo/modelo_NLLB_ShipiboKonibo_v4")

Some non-default generation parameters are set in the model config. These should go into a GenerationConfig file (https://huggingface.co/docs/transformers/generation_strategies#save-a-custom-decoding-strategy-with-your-model) instead. This warning will be raised to an exception in v4.41.
Non-default generation parameters: {'max_length': 200}


('/content/drive/MyDrive/Tesis 2/Entrenamiento modelos NMT/Shipibo Konibo/modelo_NLLB_ShipiboKonibo_v4/tokenizer_config.json',
 '/content/drive/MyDrive/Tesis 2/Entrenamiento modelos NMT/Shipibo Konibo/modelo_NLLB_ShipiboKonibo_v4/special_tokens_map.json',
 '/content/drive/MyDrive/Tesis 2/Entrenamiento modelos NMT/Shipibo Konibo/modelo_NLLB_ShipiboKonibo_v4/sentencepiece.bpe.model',
 '/content/drive/MyDrive/Tesis 2/Entrenamiento modelos NMT/Shipibo Konibo/modelo_NLLB_ShipiboKonibo_v4/added_tokens.json',
 '/content/drive/MyDrive/Tesis 2/Entrenamiento modelos NMT/Shipibo Konibo/modelo_NLLB_ShipiboKonibo_v4/tokenizer.json')

In [22]:
# ==============================================================================
# CARGA AUTOMÁTICA DEL MODELO ENTRENADO (EVITA ENTRENAR DESDE CERO)
# ==============================================================================
import os
from transformers import AutoModelForSeq2SeqLM

path_v4 = "/content/drive/MyDrive/Tesis 2/Entrenamiento modelos NMT/Shipibo Konibo/modelo_NLLB_ShipiboKonibo_v4"
path_orig = "/content/drive/MyDrive/Tesis 2/Entrenamiento modelos NMT/Shipibo Konibo/modelo_NLLB_ShipiboKonibo_v1"

if os.path.exists(path_v4):
    print(f"[INFO] Cargando modelo entrenado v4 desde: {path_v4}")
    model = AutoModelForSeq2SeqLM.from_pretrained(path_v4, local_files_only=True)
    model.to("cuda")
elif os.path.exists(path_orig):
    print(f"[INFO] Cargando modelo entrenado previo desde: {path_orig}")
    model = AutoModelForSeq2SeqLM.from_pretrained(path_orig, local_files_only=True)
    model.to("cuda")
else:
    print("[ADVERTENCIA] No se encontró ningún checkpoint en Google Drive.")
    print("[INFO] Se continuará usando el modelo base en memoria.")


[INFO] Cargando modelo entrenado v4 desde: /content/drive/MyDrive/Tesis 2/Entrenamiento modelos NMT/Shipibo Konibo/modelo_NLLB_ShipiboKonibo_v4


In [23]:
#Creación de la función de traducción
def traducir(texto):
    inputs = tokenizer(texto, return_tensors="pt").to(model.device)
    outputs = model.generate(**inputs, max_length=50)
    return tokenizer.decode(outputs[0], skip_special_tokens=True)

# VALIDACIÓN (val.csv)

In [24]:
#Cargar val

path_val = "/content/drive/MyDrive/Tesis 2/Entrenamiento modelos NMT/Shipibo Konibo/val.csv"

df_val = pd.read_csv(path_val)
df_val = df_val[["source", "target"]].dropna()

In [25]:
#Preprocesamiento de val
df_val["source_clean"] = df_val["source"].apply(limpiar_texto)
df_val["target_clean"] = df_val["target"].apply(limpiar_texto)


In [26]:
#Ejemplos de los resultados

for i in range(5):
    print("INPUT:", df_val.iloc[i]["source_clean"])
    print("REAL:", df_val.iloc[i]["target_clean"])
    print("PRED:", traducir(df_val.iloc[i]["source_clean"]))
    print("-----")

INPUT: jatianra ja dios meniti jawéki min boá jainxon dios yoina menoxontiainkobipari abaini ja joni betan joi benxoax raeananipari mia kati jake jatian moa jaskáa pekáoparikayara min ati jake dios jawéki menikin
REAL: deja allí mismo tu ofrenda ante el altar y vete antes a hacer las paces con tu hermano después vuelve y presenta tu ofrenda
PRED: y si tú lo vas a traer con tu ofrenda y lo dejas en el altar después de la fiesta haz el bien y después de la fiesta haz el bien
-----
INPUT: jain yoyo ati kirika benxoati
REAL: biblioteca del aula
PRED: biblioteca de aula
-----
INPUT: teayamawe yoyo ikashamaitian
REAL: no lo obligues
PRED: no te preocupes por el diálogo
-----
INPUT: moara jaton aká jawékibo jan jato yoixonti nete senenke
REAL: adoren al que hizo el cielo la tierra el mar y los manantiales de agua
PRED: ha llegado la hora de juzgar
-----
INPUT: rotacion itan traslacion
REAL: rotación y traslación
PRED: rotación y traslación
-----


# Métricas para val.csv

In [27]:
#Preparar listas

preds = []
refs = []

for i in range(len(df_val)):
    pred = traducir(df_val.iloc[i]["source_clean"])
    ref = df_val.iloc[i]["target_clean"]

    preds.append(pred)
    refs.append([ref])

In [28]:
#BLEU

import evaluate

bleu = evaluate.load("bleu")
print("BLEU:", bleu.compute(predictions=preds, references=refs))

BLEU: {'bleu': 0.142361937902711, 'precisions': [0.4214239342133737, 0.19620015048908954, 0.11027594728171335, 0.06695415342714481], 'brevity_penalty': 0.9056797923535218, 'length_ratio': 0.9098605969914153, 'translation_length': 23105, 'reference_length': 25394}


In [29]:
#CHRF

chrf = evaluate.load("chrf")
print("ChrF:", chrf.compute(predictions=preds, references=refs))

ChrF: {'score': 36.23148855538559, 'char_order': 6, 'word_order': 0, 'beta': 2}


In [30]:
#COMET

from comet import download_model, load_from_checkpoint

model_path = download_model("Unbabel/wmt20-comet-da")
comet_model = load_from_checkpoint(model_path)

data = []

for i in range(len(df_val)):
    data.append({
        "src": df_val.iloc[i]["source"],  # ORIGINAL
        "mt": preds[i],
        "ref": df_val.iloc[i]["target"]
    })

scores = comet_model.predict(data, batch_size=8, gpus=1)

print("COMET:", scores["system_score"])

Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

LICENSE: 0.00B [00:00, ?B/s]

README.md: 0.00B [00:00, ?B/s]

.gitattributes: 0.00B [00:00, ?B/s]

hparams.yaml:   0%|          | 0.00/437 [00:00<?, ?B/s]

...

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

config.json:   0%|          | 0.00/616 [00:00<?, ?B/s]

/usr/local/lib/python3.12/dist-packages/pytorch_lightning/core/saving.py:197: Found keys that are not in the model state dict but in the checkpoint: ['encoder.model.embeddings.position_ids']
INFO:pytorch_lightning.utilities.rank_zero:GPU available: True (cuda), used: True
INFO:pytorch_lightning.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO:pytorch_lightning.utilities.rank_zero:💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
INFO:pytorch_lightning.utilities.rank_zero:💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.
INFO:pytorch_lightning.utilities.rank_zero:You are using a CUDA device ('NVIDIA RTX PRO 6000 Blackwell Server Edition') th

COMET: -0.4964163044694202


In [31]:
# =========================
# MÉTRICA METEOR
# =========================
try:
    import evaluate
    meteor = evaluate.load("meteor")
    meteor_result = meteor.compute(predictions=preds, references=[r[0] for r in refs])
    print("METEOR:", meteor_result)
except Exception as e:
    print("Error al calcular METEOR:", e)


[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.
[nltk_data] Downloading package omw-1.4 to /root/nltk_data...


METEOR: {'meteor': 0.3411122170469636}


In [52]:
# =========================
# MÉTRICA BERTSCORE
# =========================
try:
    import evaluate
    bertscore = evaluate.load("bertscore")
    bertscore_result = bertscore.compute(predictions=preds, references=[r[0] for r in refs], lang="es")
    print("BERTScore Precision (Promedio):", sum(bertscore_result["precision"]) / len(bertscore_result["precision"]))
    print("BERTScore Recall (Promedio):", sum(bertscore_result["recall"]) / len(bertscore_result["recall"]))
    print("BERTScore F1 (Promedio):", sum(bertscore_result["f1"]) / len(bertscore_result["f1"]))
except Exception as e:
    print("Error al calcular BERTScore:", e)


BERTScore Precision (Promedio): 0.7985523815779243
BERTScore Recall (Promedio): 0.7863582399537679
BERTScore F1 (Promedio): 0.7920227885440534


In [76]:
## REENTRENAR EL MODELO, PARA REVISION DE MEJORA
# ==============================================================================
# CAMBIOS PARA EL REENTRENAMIENTO INCREMENTAL:
# 1. learning_rate=3e-5 (Se reduce a 3e-5 en comparación con el valor inicial de 5e-5)
#    para realizar un ajuste fino más conservador y evitar el olvido catastrófico de los pesos
#    ya optimizados en la fase 1.
# 2. num_train_epochs = 3 (Se mantiene en 3 épocas para el refinamiento adicional).
# ==============================================================================

training_args = TrainingArguments(
    output_dir="./results",
    per_device_train_batch_size=1,
    gradient_accumulation_steps=4,  # Lote efectivo de 4 (mayor estabilidad)
    num_train_epochs=3,
    learning_rate=3e-5,
    weight_decay=0.01,               # Evita overfitting
    warmup_ratio=0.1,                # Calentamiento del 10% de pasos
    save_strategy="epoch",
    fp16=True
)

In [77]:
##Limpiamos el entorno

import torch
torch.cuda.empty_cache()

In [78]:
import torch
import gc

del model
del trainer

gc.collect()
torch.cuda.empty_cache()

In [79]:
from transformers import AutoModelForSeq2SeqLM
import os

# ==============================================================================
# REENTRENAMIENTO INCREMENTAL RESILIENTE:
# Busca los pesos de la Fase 1 en Google Drive (v4 o original) para continuar el aprendizaje.
# Si no encuentra ninguno, cae al modelo base de Hugging Face por seguridad.
# ==============================================================================
path_v4 = "/content/drive/MyDrive/Tesis 2/Entrenamiento modelos NMT/Shipibo Konibo/modelo_NLLB_ShipiboKonibo_v4"
path_orig = "/content/drive/MyDrive/Tesis 2/Entrenamiento modelos NMT/Shipibo Konibo/modelo_nllb_ShipiboKonibo_v1"

if os.path.exists(path_v4):
    print(f"[INFO] Cargando pesos de Fase 1 (v4) para reentrenamiento: {path_v4}")
    model = AutoModelForSeq2SeqLM.from_pretrained(path_v4, local_files_only=True)
elif os.path.exists(path_orig):
    print(f"[INFO] Cargando pesos de Fase 1 (original) para reentrenamiento: {path_orig}")
    model = AutoModelForSeq2SeqLM.from_pretrained(path_orig, local_files_only=True)
else:
    print("[ADVERTENCIA] No se encontró modelo previo. Cargando modelo base desde cero...")
    model = AutoModelForSeq2SeqLM.from_pretrained("facebook/nllb-200-distilled-600M")

model.to("cuda")
# =========================
# REDIMENSIONAR EMBEDDINGS
# =========================
model.resize_token_embeddings(len(tokenizer))


[INFO] Cargando pesos de Fase 1 (v4) para reentrenamiento: /content/drive/MyDrive/Tesis 2/Entrenamiento modelos NMT/Shipibo Konibo/modelo_NLLB_ShipiboKonibo_v4


Embedding(256268, 1024, padding_idx=1)

In [80]:
print(tokenized_dataset)
print(tokenized_dataset.column_names)

ejemplo = tokenized_dataset[0]

print(ejemplo.keys())
print("Tiene labels:", "labels" in ejemplo)

Dataset({
    features: ['input_ids', 'attention_mask', 'labels'],
    num_rows: 14728
})
['input_ids', 'attention_mask', 'labels']
dict_keys(['input_ids', 'attention_mask', 'labels'])
Tiene labels: True


In [81]:
##Creamos del trainer despues de limpiar memoria

from transformers import Trainer
from transformers import TrainingArguments
from transformers import DataCollatorForSeq2Seq

data_collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer,
    model=model,
    return_tensors="pt"
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset,
    data_collator=data_collator
)

In [82]:
batch = data_collator([
    tokenized_dataset[0],
    tokenized_dataset[1]
])

for k,v in batch.items():
    print(k, v.shape)

input_ids torch.Size([2, 64])
attention_mask torch.Size([2, 64])
labels torch.Size([2, 64])


In [83]:
#Realizamos el entrenamiento despues de los siguientes cambios de parametros

trainer.train()

Step,Training Loss
500,0.464700
1000,0.453200
1500,0.443200
2000,0.426200
2500,0.424500
3000,0.401300
3500,0.390100
4000,0.357800
4500,0.338100
5000,0.342600


Some non-default generation parameters are set in the model config. These should go into a GenerationConfig file (https://huggingface.co/docs/transformers/generation_strategies#save-a-custom-decoding-strategy-with-your-model) instead. This warning will be raised to an exception in v4.41.
Non-default generation parameters: {'max_length': 200}
Some non-default generation parameters are set in the model config. These should go into a GenerationConfig file (https://huggingface.co/docs/transformers/generation_strategies#save-a-custom-decoding-strategy-with-your-model) instead. This warning will be raised to an exception in v4.41.
Non-default generation parameters: {'max_length': 200}
Some non-default generation parameters are set in the model config. These should go into a GenerationConfig file (https://huggingface.co/docs/transformers/generation_strategies#save-a-custom-decoding-strategy-with-your-model) instead. This warning will be raised to an exception in v4.41.
Non-default generation 

TrainOutput(global_step=11046, training_loss=0.36678131019941773, metrics={'train_runtime': 2139.5885, 'train_samples_per_second': 20.651, 'train_steps_per_second': 5.163, 'total_flos': 5984459358732288.0, 'train_loss': 0.36678131019941773, 'epoch': 3.0})

In [84]:
##Guardar 2da version

trainer.save_model("/content/drive/MyDrive/Tesis 2/Entrenamiento modelos NMT/Shipibo Konibo/modelo_nllb_ShipiboKonibo_v4_retrained")
tokenizer.save_pretrained("/content/drive/MyDrive/Tesis 2/Entrenamiento modelos NMT/Shipibo Konibo/modelo_nllb_ShipiboKonibo_v4_retrained")

Some non-default generation parameters are set in the model config. These should go into a GenerationConfig file (https://huggingface.co/docs/transformers/generation_strategies#save-a-custom-decoding-strategy-with-your-model) instead. This warning will be raised to an exception in v4.41.
Non-default generation parameters: {'max_length': 200}


('/content/drive/MyDrive/Tesis 2/Entrenamiento modelos NMT/Shipibo Konibo/modelo_nllb_ShipiboKonibo_v4_retrained/tokenizer_config.json',
 '/content/drive/MyDrive/Tesis 2/Entrenamiento modelos NMT/Shipibo Konibo/modelo_nllb_ShipiboKonibo_v4_retrained/special_tokens_map.json',
 '/content/drive/MyDrive/Tesis 2/Entrenamiento modelos NMT/Shipibo Konibo/modelo_nllb_ShipiboKonibo_v4_retrained/sentencepiece.bpe.model',
 '/content/drive/MyDrive/Tesis 2/Entrenamiento modelos NMT/Shipibo Konibo/modelo_nllb_ShipiboKonibo_v4_retrained/added_tokens.json',
 '/content/drive/MyDrive/Tesis 2/Entrenamiento modelos NMT/Shipibo Konibo/modelo_nllb_ShipiboKonibo_v4_retrained/tokenizer.json')

In [88]:
# Carga y preparación del dataset de validación para evaluar el modelo después del reentrenamiento
print("Generando predicciones sobre el conjunto de validación con el modelo reentrenado...")

#Probar ejemplos
val_dataset = Dataset.from_pandas(df_val)
for i in range(5):
    print("INPUT:", df_val.iloc[i]["source_clean"])
    print("REAL:", val_dataset[i]["target"])
    print("PRED:", traducir(df_val.iloc[i]["source_clean"]))
    print("-----")


Generando predicciones sobre el conjunto de validación con el modelo reentrenado...
INPUT: jatianra ja dios meniti jawéki min boá jainxon dios yoina menoxontiainkobipari abaini ja joni betan joi benxoax raeananipari mia kati jake jatian moa jaskáa pekáoparikayara min ati jake dios jawéki menikin
REAL: deja allí mismo tu ofrenda ante el altar y vete antes a hacer las paces con tu hermano después vuelve y presenta tu ofrenda
PRED: tú pues lo vas a hacer cuando hay que ir a ofrecerle el sacrificio y después de reunirse con él te darás el segundo parte de tu ofrenda
-----
INPUT: jain yoyo ati kirika benxoati
REAL: biblioteca del aula
PRED: biblioteca de aula
-----
INPUT: teayamawe yoyo ikashamaitian
REAL: no lo obligues
PRED: no usar la tentación cuando hablan
-----
INPUT: moara jaton aká jawékibo jan jato yoixonti nete senenke
REAL: adoren al que hizo el cielo la tierra el mar y los manantiales de agua
PRED: aquí viene el juicio
-----
INPUT: rotacion itan traslacion
REAL: rotación y trasl

In [89]:
#Generar predicciones

preds = []
refs = []

for i in range(len(val_dataset)):
    pred = traducir(df_val.iloc[i]["source_clean"])
    ref = val_dataset[i]["target"]

    preds.append(pred)
    refs.append([ref])

In [90]:
##Aplicamos nuevamente las métricas


##BLEU

import evaluate

bleu = evaluate.load("bleu")
bleu.compute(predictions=preds, references=refs)


{'bleu': 0.1198926926085625,
 'precisions': [0.3836717605191543,
  0.1633097441480675,
  0.08701247277766778,
  0.049926658336502415],
 'brevity_penalty': 0.9334073751969321,
 'length_ratio': 0.9355293564686068,
 'translation_length': 23885,
 'reference_length': 25531}

In [91]:
##CHRF

chrf = evaluate.load("chrf")

chrf_result = chrf.compute(predictions=preds, references=refs)

print("ChrF:", chrf_result)

ChrF: {'score': 34.19335531586359, 'char_order': 6, 'word_order': 0, 'beta': 2}


In [92]:
##COMET - Evaluación

from comet import download_model, load_from_checkpoint

model_path = download_model("Unbabel/wmt22-comet-da")
comet_model = load_from_checkpoint(model_path)

data = []

for i in range(len(val_dataset)):
    data.append({
        "src": df_val.iloc[i]["source_clean"],
        "mt": preds[i],
        "ref": val_dataset[i]["target"]
    })

scores = comet_model.predict(data, batch_size=8)

print("COMET:", scores["system_score"])

Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

README.md: 0.00B [00:00, ?B/s]

LICENSE: 0.00B [00:00, ?B/s]

.gitattributes: 0.00B [00:00, ?B/s]

hparams.yaml:   0%|          | 0.00/567 [00:00<?, ?B/s]

checkpoints/model.ckpt:   0%|          | 0.00/2.32G [00:00<?, ?B/s]

INFO:pytorch_lightning.utilities.migration.utils:Lightning automatically upgraded your loaded checkpoint from v1.8.3.post1 to v2.6.5. To apply the upgrade to your files permanently, run `python -m pytorch_lightning.utilities.upgrade_checkpoint ../root/.cache/huggingface/hub/models--Unbabel--wmt22-comet-da/snapshots/2760a223ac957f30acfb18c8aa649b01cf1d75f2/checkpoints/model.ckpt`
/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/pytorch_lightning/core/saving.py:197: Found keys that are not in the model state dict but in the checkpoint: ['encoder.model.embeddings.position_ids']
INFO:pytorch_lightning.utilities.rank_zero:GPU available: True (cuda), used: True
INFO:pytorch_lightning.utilities.rank_zero:TPU available: Fals

COMET: 0.6187528273549047


In [93]:
# =========================
# MÉTRICA METEOR (VALIDACIÓN - REENTRENADO)
# =========================
try:
    import evaluate
    meteor = evaluate.load("meteor")
    meteor_result = meteor.compute(predictions=preds, references=[r[0] for r in refs])
    print("METEOR (Validación - Reentrenado):", meteor_result)
except Exception as e:
    print("Error al calcular METEOR en validación con reentrenamiento:", e)


[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package omw-1.4 to /root/nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!


METEOR (Validación - Reentrenado): {'meteor': 0.3057713668455072}


In [94]:
# =========================
# MÉTRICA BERTSCORE (VALIDACIÓN - REENTRENADO)
# =========================
try:
    import evaluate
    bertscore = evaluate.load("bertscore")
    bertscore_result = bertscore.compute(predictions=preds, references=[r[0] for r in refs], lang="es")
    print("BERTScore Precision (Validación - Reentrenado - Promedio):", sum(bertscore_result["precision"]) / len(bertscore_result["precision"]))
    print("BERTScore Recall (Validación - Reentrenado - Promedio):", sum(bertscore_result["recall"]) / len(bertscore_result["recall"]))
    print("BERTScore F1 (Validación - Reentrenado - Promedio):", sum(bertscore_result["f1"]) / len(bertscore_result["f1"]))
except Exception as e:
    print("Error al calcular BERTScore en validación con reentrenamiento:", e)


/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


BERTScore Precision (Validación - Reentrenado - Promedio): 0.7848248391097036
BERTScore Recall (Validación - Reentrenado - Promedio): 0.7746382544119678
BERTScore F1 (Validación - Reentrenado - Promedio): 0.7793352779067774


In [95]:
##PROBAMOS EL TEST.CSV

path_test = "/content/drive/MyDrive/Tesis 2/Entrenamiento modelos NMT/Shipibo Konibo/test.csv"

df_test = pd.read_csv(path_test)
df_test = df_test[["source", "target"]].dropna()

In [96]:
#Preprocesamiento de test
df_test["source_clean"] = df_test["source"].apply(limpiar_texto)
df_test["target_clean"] = df_test["target"].apply(limpiar_texto)


In [97]:
from datasets import Dataset

# ==============================================================================
# CREACIÓN DEL DATASET DE PRUEBA:
# Corregimos la NameError asegurando que se carga a partir de df_test (y no del df de entrenamiento).
# ==============================================================================
df_test_model = df_test[["source_clean", "target_clean"]].rename(
    columns={
        "source_clean": "source",
        "target_clean": "target"
    }
)
test_dataset = Dataset.from_pandas(df_test_model)
print("Dataset de prueba cargado con éxito:", test_dataset)


Dataset de prueba cargado con éxito: Dataset({
    features: ['source', 'target'],
    num_rows: 1841
})


In [98]:
preds = []
refs = []

for i in range(len(test_dataset)):
    pred = traducir(df_test.iloc[i]["source_clean"])
    ref = test_dataset[i]["target"]

    preds.append(pred)
    refs.append([ref])

In [99]:
##BLEU

bleu.compute(predictions=preds, references=refs)

{'bleu': 0.12741701188482832,
 'precisions': [0.38827545515846257,
  0.16594325398638463,
  0.08879134035017708,
  0.053412950586172896],
 'brevity_penalty': 0.9637141502956803,
 'length_ratio': 0.9643568380410485,
 'translation_length': 23728,
 'reference_length': 24605}

In [100]:
##CHRF
chrf.compute(predictions=preds, references=refs)

{'score': 34.86367317931408, 'char_order': 6, 'word_order': 0, 'beta': 2}

In [101]:
##COMET - Evaluación

from comet import download_model, load_from_checkpoint

model_path = download_model("Unbabel/wmt22-comet-da")
comet_model = load_from_checkpoint(model_path)

data = []

for i in range(len(test_dataset)):
    data.append({
        "src": df_test.iloc[i]["source_clean"],
        "mt": preds[i],
        "ref": test_dataset[i]["target"]
    })

scores = comet_model.predict(data, batch_size=8)

print("COMET:", scores["system_score"])

Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

INFO:pytorch_lightning.utilities.migration.utils:Lightning automatically upgraded your loaded checkpoint from v1.8.3.post1 to v2.6.5. To apply the upgrade to your files permanently, run `python -m pytorch_lightning.utilities.upgrade_checkpoint ../root/.cache/huggingface/hub/models--Unbabel--wmt22-comet-da/snapshots/2760a223ac957f30acfb18c8aa649b01cf1d75f2/checkpoints/model.ckpt`
/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/pytorch_lightning/core/saving.py:197: Found keys that are not in the model state dict but in the checkpoint: ['encoder.model.embeddings.position_ids']
INFO:pytorch_lightning.utilities.rank_zero:GPU available: True (cuda), used: True
INFO:pytorch_lightning.utilities.rank_zero:TPU available: Fals

COMET: 0.6180698368809652


In [102]:
# =========================
# MÉTRICA METEOR
# =========================
try:
    import evaluate
    meteor = evaluate.load("meteor")
    meteor_result = meteor.compute(predictions=preds, references=[r[0] for r in refs])
    print("METEOR:", meteor_result)
except Exception as e:
    print("Error al calcular METEOR:", e)


[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package omw-1.4 to /root/nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!


METEOR: {'meteor': 0.3155740986639407}


In [103]:
# =========================
# MÉTRICA BERTSCORE
# =========================
try:
    import evaluate
    bertscore = evaluate.load("bertscore")
    bertscore_result = bertscore.compute(predictions=preds, references=[r[0] for r in refs], lang="es")
    print("BERTScore Precision (Promedio):", sum(bertscore_result["precision"]) / len(bertscore_result["precision"]))
    print("BERTScore Recall (Promedio):", sum(bertscore_result["recall"]) / len(bertscore_result["recall"]))
    print("BERTScore F1 (Promedio):", sum(bertscore_result["f1"]) / len(bertscore_result["f1"]))
except Exception as e:
    print("Error al calcular BERTScore:", e)


/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


BERTScore Precision (Promedio): 0.7839431361230025
BERTScore Recall (Promedio): 0.7780100805872099
BERTScore F1 (Promedio): 0.7806082683241542
